In [3]:
import os
import sys
os.chdir("..") 
sys.path.append(os.getcwd())

In [4]:
import pandas as pd
import numpy as np
from sklearn import linear_model
from sklearn.linear_model import RidgeCV
from sklearn.neural_network import MLPRegressor
from sklearn.svm import LinearSVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import BayesianRidge, Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from tpot import TPOTRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline, make_union
from sklearn.preprocessing import PolynomialFeatures
from tpot.builtins import StackingEstimator
from tpot.export_utils import set_param_recursive
from sklearn.metrics import mean_squared_error, r2_score
from AutoML_Tpot.data_processing import  data_processing
from AutoML_Tpot.normalize import normalize_predictions_by_month

In [5]:
print(f"Répertoire actuel après retour en arrière : {os.getcwd()}")

Répertoire actuel après retour en arrière : /workspaces/AutoMLTPOT


In [6]:

file_path =  '/workspaces/AutoMLTPOT/data/kable_data.xlsx'

# Charger les feuilles Excel
ws = pd.read_excel(file_path, sheet_name="Massive for learning")
ws_test = pd.read_excel(file_path, sheet_name="Massive")
# Génère des nombres aléatoires et les formate à 3 décimales
ws['tabet'] = np.random.uniform(0.001, 0.01, size=len(ws))
# Nom de la colonne à déplacer
colonne_a_deplacer = 'tabet'
# Créer la nouvelle liste de colonnes
nouvelles_colonnes = [colonne_a_deplacer] + [col for col in ws.columns if col != colonne_a_deplacer]
# Réindexer le DataFrame
ws = ws[nouvelles_colonnes]

In [7]:
X , Y = data_processing (ws, ws_test)
X_features = X[:, 2:]
y_target = X[:, 0]
# Forcer la conversion en float, gérer erreurs (non numériques -> NaN puis remplacées)
X_features = np.nan_to_num(X_features.astype(float))
y_target = pd.to_numeric(y_target, errors='coerce')

valid_idx = ~np.isnan(y_target)
X_features = X_features[valid_idx]
y_target = y_target[valid_idx]
X_train, X_test, y_train, y_test = train_test_split(X_features, y_target, test_size= .25)

In [8]:
reg = TPOTRegressor(verbosity=2, population_size=50, generations=1, random_state=35)
reg.fit(X_train, y_train)
print(reg.score(X_test, y_test))
#save the model in top_boston.py
reg.export('/workspaces/AutoMLTPOT/python_boston/top_bostonkable.py')
print('Exported to python_boston/top_bostonkable.py')

is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor


/home/codespace/.local/lib/python3.12/site-packages/sklearn/base.py:1230: FutureWarning: passing a class to None is deprecated and will be removed in 1.8. Use an instance of the class instead.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/base.py:1270: FutureWarning: passing a class to None is deprecated and will be removed in 1.8. Use an instance of the class instead.
  warnings.warn(


is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor


Version 0.12.2 of tpot is outdated. Version 1.1.0 was released Thursday July 03, 2025.


Optimization Progress:   0%|          | 0/100 [00:00<?, ?pipeline/s]


Generation 1 - Current best internal CV score: -6.381133968764561e-06

Best pipeline: XGBRegressor(input_matrix, learning_rate=0.001, max_depth=4, min_child_weight=7, n_estimators=100, n_jobs=1, objective=reg:squarederror, subsample=0.7500000000000001, verbosity=0)
-6.821028998810857e-06
Exported to python_boston/top_bostonkable.py


In [9]:
testing_features = Y
# Vérification des formes des données
print("Shape of training features:", X_features.shape)
print("Shape of testing features:", testing_features.shape)

Shape of training features: (1339, 18)
Shape of testing features: (1339, 18)


In [ ]:
# Après tpot 
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

In [10]:
# Average CV score on the training set was: -6.381133968764561e-06
exported_pipeline = XGBRegressor(learning_rate=0.001, max_depth=4, min_child_weight=7, n_estimators=100, n_jobs=1, objective="reg:squarederror", subsample=0.7500000000000001, verbosity=0)
# Fix random state in exported estimator
if hasattr(exported_pipeline, 'random_state'):
    setattr(exported_pipeline, 'random_state', 35)

exported_pipeline.fit(X_features, y_target)

predictions = exported_pipeline.predict(X_features)
# Clipping pour forcer les valeurs entre 0 et 1
predictions = np.clip(predictions, 0, 1)

# Calcul des métriques
n_eval = min(len(predictions), len(y_target))
mse = mean_squared_error(y_target[:n_eval], predictions[:n_eval])
r2 = r2_score(y_target[:n_eval], predictions[:n_eval])

results = {"MSE": mse, "R2": r2, "Predictions": predictions}
df_actual = pd.DataFrame(y_target[:n_eval], columns=["Réel"])
df_pred = pd.DataFrame(predictions[:n_eval], columns=["Prédit"])
desc_actual = df_actual["Réel"].describe()
desc_pred = df_pred["Prédit"].describe()
stats_df = pd.concat([desc_actual, desc_pred], axis=1)
stats_df.columns = ["Statistiques Факт", "Statistiques Прогноз"]
print(stats_df)
#print(results)

# Prédictions sur les données de test
predictions = np.clip(exported_pipeline.predict(testing_features), 0.0001, 1)
#predictions = exported_pipeline.predict(testing_features)


       Statistiques Факт  Statistiques Прогноз
count        1339.000000           1339.000000
mean            0.005478              0.005478
std             0.002544              0.000027
min             0.001009              0.005400
25%             0.003303              0.005458
50%             0.005468              0.005478
75%             0.007637              0.005496
max             0.009997              0.005573


In [13]:
print(f"Répertoire actuel après retour en arrière : {os.getcwd()}")

Répertoire actuel après retour en arrière : /workspaces/AutoMLTPOT


In [15]:
# Normaliser les prédictions
normalized_results = normalize_predictions_by_month(ws_test, predictions)

# Exporter les résultats normalisés dans un fichier Excel
output_file_path = '/workspaces/AutoMLTPOT/python_boston/results.xlsx'
normalized_results.to_excel(output_file_path, index=False,engine='openpyxl')

print(f"Les résultats normalisés ont été exportés vers {output_file_path}")

Сумма прогнозов по месяцам (полные месяцы) :
Mois
2022-01    1.0
2022-02    1.0
2022-03    1.0
2022-04    1.0
2022-05    1.0
2022-06    1.0
2022-07    1.0
2022-08    1.0
2022-09    1.0
2022-10    1.0
2022-11    1.0
2022-12    1.0
2023-01    1.0
2023-02    1.0
2023-03    1.0
2023-04    1.0
2023-05    1.0
2023-06    1.0
2023-07    1.0
2023-08    1.0
2023-09    1.0
2023-10    1.0
2023-11    1.0
2023-12    1.0
2024-01    1.0
2024-02    1.0
2024-03    1.0
2024-04    1.0
2024-05    1.0
2024-06    1.0
2024-07    1.0
2024-08    1.0
2024-09    1.0
2024-10    1.0
2024-11    1.0
2024-12    1.0
2025-01    1.0
2025-02    1.0
2025-03    1.0
2025-04    1.0
2025-05    1.0
2025-06    1.0
2025-07    1.0
2025-08    1.0
Freq: M, Name: Prédiction, dtype: float32
Les résultats normalisés ont été exportés vers /workspaces/AutoMLTPOT/python_boston/results.xlsx
